In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm

def create_dataset():
    # Пути к данным
    base_dir = "common_voice_processed"
    audio_dir = os.path.join(base_dir, "audio/mixed")
    meta_dir = os.path.join(base_dir, "metadata/mixed")
    dataset_dir = os.path.join(base_dir, "dataset")
    os.makedirs(dataset_dir, exist_ok=True)

    # Сбор всех метаданных
    metadata = []
    for meta_file in tqdm(os.listdir(meta_dir), desc="Обработка метаданных"):
        if not meta_file.endswith('.json'):
            continue
            
        with open(os.path.join(meta_dir, meta_file), 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Добавляем путь к аудиофайлу
            audio_file = meta_file.replace('.json', '.wav')
            data['audio_path'] = os.path.join(audio_dir, audio_file)
            
            # Проверяем существование аудиофайла
            if not os.path.exists(data['audio_path']):
                continue
                
            metadata.append(data)

    # Создание основной таблицы
    dataset = []
    speakers = set()
    
    for item in tqdm(metadata, desc="Формирование датасета"):
        # Основная информация
        row = {
            'id': item['id'],
            'audio_path': item['audio_path'],
            'pattern': item['pattern'],
            'duration': item['duration'],
            'with_noise': item.get('with_noise', False),
            'overlap_ratio': item.get('overlap_ratio', 0)
        }
        
        # Информация о спикерах
        for i, comp in enumerate(item['components'], 1):
            row[f'speaker_{i}_id'] = comp['speaker']
            row[f'speaker_{i}_duration'] = comp['duration']
            speakers.add(comp['speaker'])
        
        dataset.append(row)

    # Сохранение датасета
    df = pd.DataFrame(dataset)
    
    # Основной CSV файл
    csv_path = os.path.join(dataset_dir, "dataset.csv")
    df.to_csv(csv_path, index=False, encoding='utf-8')
    
    # JSON-версия
    json_path = os.path.join(dataset_dir, "dataset.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    
    # Список спикеров
    speakers_path = os.path.join(dataset_dir, "speakers.json")
    with open(speakers_path, 'w', encoding='utf-8') as f:
        json.dump(sorted(list(speakers)), f, indent=2, ensure_ascii=False)

    # Создание README
    readme = f"""# Dataset Description

## General Information
- Total samples: {len(df)}
- Unique speakers: {len(speakers)}
- Average duration: {df['duration'].mean():.2f} sec
- Patterns: {df['pattern'].unique().tolist()}
- With noise: {df['with_noise'].sum()} samples

## Files Structure
- `dataset.csv` - Main dataset table
- `dataset.json` - Full metadata
- `speakers.json` - List of unique speakers
- `audio/` - Directory with mixed audio files
"""
    with open(os.path.join(dataset_dir, "README.md"), 'w', encoding='utf-8') as f:
        f.write(readme)

    print(f"\nДатасет успешно создан в {dataset_dir}")
    print(f"Всего записей: {len(df)}")
    print(f"Уникальных спикеров: {len(speakers)}")

if __name__ == "__main__":
    create_dataset()

Формирование датасета: 100%|██████████| 7441/7441 [00:00<00:00, 465144.73it/s]



Датасет успешно создан в common_voice_processed\dataset
Всего записей: 7441
Уникальных спикеров: 183
